# M1B1 - Classification automatique des intentions clients - Modernisation d'un pipeline NLP chez Finova

## Étape 1 : Reproduire la baseline TF-IDF en sklearn


Reproduire le PoC du stagiaire (TF-IDF + régression logistique), proprement.

- Analyser la base (informations basiques sur les données)
- Vectorisation TF-IDF.
- LogisticRegression sur TF-IDF → modèle historique à reproduire.
- Métriques : accuracy, F1 macro, F1 weighted.
- Analyse d'erreurs : 5–10 mauvaises prédictions, type d'erreur (synonyme, reformulation, ambiguïté).

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics import accuracy_score

### 1.1. Chargement du jeu de données

In [7]:
from datasets import load_dataset

# =============================
# 0. Chargement du jeu de données
# =============================

# Chargement complet (train + test)
dataset = load_dataset("mteb/banking77")

# Affichage de la structure du dataset
print(dataset)

# Affichage d'un exemple de données
print("\nDataset sample")
print(dataset["train"][0])

# Distribution du dataset
df = pd.DataFrame({"label": dataset["train"]["label"]})
print("\nDataset Distribution")
print(df["label"].value_counts(normalize=True))

# ratio de désequilibre
max_count = max(df["label"].value_counts())
min_count = min(df["label"].value_counts())

imbalance_ratio = max_count / min_count
print("Imbalance ratio:", imbalance_ratio)


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})

Dataset sample
{'text': 'I am still waiting on my card?', 'label': 11, 'label_text': 'card_arrival'}

Dataset Distribution
label
15    0.018713
28    0.018213
6     0.018113
75    0.018013
19    0.017712
        ...   
41    0.008206
18    0.006104
10    0.005904
72    0.004103
23    0.003502
Name: proportion, Length: 77, dtype: float64
Imbalance ratio: 5.3428571428571425


**Note :**
La valeur *Imbalance ratio* montre que le dataset est assez déséquilibré (certaines classes majoritaires par rapport à d'autres)
Par conséquent :
   - L'*accuracy* seule ne permet pas de bien évalué un modèle sur ce dataset
   - Les *precision* et *recall* sont trop détaillés pour évaluation globale
   - Le *F1-weighted* constitue la meilleure metrique d'évaluation d'un modèle

In [9]:
# Conversion dans pandas, plus pratique
df_train = dataset["train"].to_pandas()
df_test  = dataset["test"].to_pandas()

# On sépare le jeu de données, X = texts, y = labels
X_train = df_train["text"]
y_train = df_train["label"]

X_test = df_test["text"]
y_test = df_test["label"]
y_test_label_text = df_test["label_text"]

### 1.2. Vectorisation

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# =============================
# Vectorisation
# =============================

# intialisation du vectorizer
vectorizer = TfidfVectorizer()
# tranformation : Learn vocabulary and idf, return document-term matrix
X_tfidf = vectorizer.fit_transform(X_train)

# taille de la matrice (nb lignes = phrases, nb col = vocabulaire)
print(X_tfidf.shape)
print(vectorizer.get_feature_names_out()[:20])

(9993, 2319)
['00' '000' '10' '100' '13' '16' '18' '1818' '1l' '20' '200' '2018' '30'
 '3d' '40' '45' '50' '500' '5x' '60']


Récap des étapes :
"text" → vecteur numérique → modèle ML possible

### 1.3. LogisticRegression sur TF-IDF

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# =============================
# 1. Pipeline
# =============================
# Le pipeline : raw text → word counts → TF-IDF → model (LogicticRegression)
text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', LogisticRegression()),  # Modèle Regression Logistique
])


# =============================
# 2. Entrainement
# =============================

text_clf.fit(X_train, y_train)


# =============================
# 3. Predictions et Confiance
# =============================
# Prédiction
predicted = text_clf.predict(X_test)

# Probabilités pour le score de confiance
probas = text_clf.predict_proba(X_test)
# score de confiance
confidence = probas.max(axis=1)


### 1.4. Métriques Accuracy

In [12]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# =============================
# 4. Metrics globales
# =============================

# Accuracy, F1 macro, F1 weighted
accuracy = text_clf.score(X_test, y_test) # Evalue la performance du modèle en comparant le résultat des prédictions et la classe de test
f1_macro = f1_score(y_test, predicted, average='macro')
f1_weighted = f1_score(y_test, predicted, average='weighted')

print(f"Accuracy sur le test set : {accuracy:.8f}") # formattage accuracy avec 8 décimales
print(f"F1 Macro sur le test set : {f1_macro:.8f}") # formattage accuracy avec 8 décimales
print(f"F1 Weighted sur le test set : {f1_weighted:.8f}") # formattage accuracy avec 8 décimales

# print("\nRapport de classification détaillé :")
# print(classification_report(y_test, predicted)) # rapport détaillé de performance du modèle

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "F1 Macro", "F1 Weighted"],
    "Value": [accuracy, f1_macro, f1_weighted]
})

# =============================
# 5. Classification report structuré
# =============================
report_dict = classification_report(y_test, predicted, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

# =============================
# 6. Confusion matrix
# =============================
labels = sorted(set(y_test))
cm = confusion_matrix(y_test, predicted, labels=labels)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)

# =============================
# 7. Dataset de prédictions 
# =============================
results_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "predicted_label": predicted,
    "confidence": confidence
})

results_df["correct"] = results_df["true_label"] == results_df["predicted_label"]

# =============================
# 8. Dataset des labels 
# =============================

labels_df = (
    df_test[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
    .rename(columns={
        "label": "label_id",
        "label_text": "label_text"
    })
)


# =============================
# 9. Export Excel multi-feuilles
# =============================
output_file = "eval_model_nlp.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    metrics_df.to_excel(writer, sheet_name="Metrics_globales", index=False)
    report_df.to_excel(writer, sheet_name="Classification_report")
    cm_df.to_excel(writer, sheet_name="Confusion_matrix")
    results_df.to_excel(writer, sheet_name="Predictions", index=False)
    labels_df.to_excel(writer, sheet_name="Labels_ref", index=False)

print(f"\n=====> Export terminé : {output_file}")



Accuracy sur le test set : 0.87776333
F1 Macro sur le test set : 0.87769756
F1 Weighted sur le test set : 0.87772974

=====> Export terminé : eval_model_nlp.xlsx


### 1.5. Analyse d'erreur

Les résultats des prédictions sont présentés dans l'onglet `Prédiction` du fichier `eval_model_nlp.xlsx.
Parmis les erreurs de prédiction, on distingue les cas suivants :
   - les erreurs de prédiction sont souvent liées à une ambiguité du texte et le fait que 2 labels peuvent avoir une significaiton très proche. (exemples lignes 7, 13, 34, )
   - Parfois le texte n'est pas assez explicite (ex: ligne 95)
   - certaines erreurs relèvent d'une proximité sémantique (celà joue parfois à un mot : 104 vs 108)

La matrice de confusion montre ue certaines classes sont plus difficiles à prédire que d'autres. Par exemple la classe 69 comporte 9 erreurs de préditcion vers la classe 74 : 
   - 69 - *verify_my_identity*
   - 74 - *why_verify_identity*
On constate que ces classes sont très proches sémantiquement, ce qui peut expliquer un nombre important d'erreur.

En conclusion le modèle est globalement bon (accuracy, F1_weighted à 88 %) mais sujet à erreurs lorsque la sémantique de certains labels devient trop proche. Plusieurs pistes sont envisageables pour améliorer ces prédictions : changer de modèle (LinearSVC, RandomForestClassifier), à tester, où entrainement spécifiquement le modèle dans un contexte métier particulier.


## Étape 2 : Tester d'autres classifieurs sur la même couche TF-IDF

La régression logistique n'est qu'un choix parmi d'autres. Avant de présumer qu'il faut sauter sur du neuronal, vérifier qu'on ne gagne pas simplement en changeant de classifieur. Sur les mêmes features TF-IDF que l'étape 1, entraîner et évaluer :

- LinearSVC ;
- RandomForestClassifier ;
- ou voir la documentation de sklearn.

In [15]:
# Build the pipeline: raw text → word counts → TF-IDF → Naive Bayes

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
# les modèles à comparer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(random_state=42),
    "Naive Bayes": MultinomialNB()
    # "Decision Tree": DecisionTreeClassifier(random_state=42),
    # "KNN": KNeighborsClassifier(),
    # "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    # "Naive Bayes": GaussianNB(),
    # "Neural Network": MLPClassifier(max_iter=1000, random_state=42),  
}

results = []

# Itération sur la liste des classifier à tester
for name, clf in classifiers.items():
    # clf.fit(X_train, y_train)      # entrainement
    # y_pred = clf.predict(X_test)   # prediction

    # Construction du Pipeline
    clf = Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', clf),  # Modèle Regression Logistique        
    ])
    
    # Entrainement et évaluation du modèle
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # Métriques
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        "Classifier": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Macro": f1_macro,
        "F1-Weighted": f1_weighted
    })

# Afficher les résultats sous forme de tableau, on trie par F1-Weighted
results_df = pd.DataFrame(results)
print(results_df.sort_values(by="F1-Weighted", ascending=False))



            Classifier  Accuracy  Precision    Recall  F1-Macro  F1-Weighted
2                  SVM  0.895969   0.903111  0.895969  0.896416     0.896467
0  Logistic Regression  0.877763   0.886508  0.877763  0.877698     0.877730
1        Random Forest  0.867035   0.872823  0.867035  0.866547     0.866614
3          Naive Bayes  0.787711   0.816449  0.787711  0.772075     0.772158


C:\Users\A453784\Simplon\M1B1_classification_auto\m1b1_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Étape 3 : Validation croisée (Cross validation)

Tous vos chiffres jusqu'ici reposent sur un seul split. Sans validation croisée (CV), vous ne savez pas si la différence entre deux modèles est réelle ou du bruit de découpage et votre choix pour la suite pourrait être un coup de chance.

Implémenter une validation croisée afin d'évaluer les modèles implémentés.

In [17]:
import pandas as pd
import numpy as np
import time
from tqdm.notebook import tqdm # pour afficher des barres de progression

# fonction permettant d'effectuer la validation croisée
from sklearn.model_selection import cross_val_score, cross_validate
# from sklearn.model_selection import KFold

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# les modèles à comparer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    # "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(random_state=42),
    # "KNN": KNeighborsClassifier(),
    # "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    # "Naive Bayes": GaussianNB(),
    # "Neural Network": MLPClassifier(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB()
}

# Reconstitution du jeu de données
X = pd.concat([X_train, X_test])
y = pd.concat([y_train, y_test])

results = []

# Itération sur la liste des classifier à tester
for name, clf in tqdm(classifiers.items(), desc="Progression"):
    start_time = time.time()  # Temps de début
    
    # Construction du Pipeline
    clf = Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', clf),  # Modèle Regression Logistique        
    ])

    
    scoring = {
        'accuracy': 'accuracy',
        'f1_macro': 'f1_macro',
        'f1_weighted': 'f1_weighted',
        'precision_macro': 'precision_macro',
        'recall_macro': 'recall_macro'
    }
    
    cv_results = cross_validate(clf, X, y, cv=5, scoring=scoring)

    # Effectuer une validation croisée à 5 folds
    # scores = cross_val_score(clf, X, y, cv=5)
    # accuracy = cross_val_score(clf, X, y, scoring='accuracy')
    # f1_macro = cross_val_score(clf, X, y, scoring='f1_macro')
    # f1_weighted = cross_val_score(clf, X, y, scoring='f1_weighted')
    
    # Calculer la moyenne des scores
    # mean_score = scores.mean()
    
    # Calculer l'écart type des scores
    # std_dev = scores.std()

    elapsed = time.time() - start_time  # Durée
    

    # Stockage des résultats moyens
    results.append({
        "Classifier": name,
        "Accuracy_mean": cv_results["test_accuracy"].mean(),
        "Accuracy_std": cv_results["test_accuracy"].std(),
        "F1_macro_mean": cv_results["test_f1_macro"].mean(),
        "F1_macro_std": cv_results["test_f1_macro"].std(),
        "F1_weighted_mean": cv_results["test_f1_weighted"].mean(),
        "F1_weighted_std": cv_results["test_f1_weighted"].std(),
        "Precision_macro_mean": cv_results["test_precision_macro"].mean(),
        "Recall_macro_mean": cv_results["test_recall_macro"].mean(),
        "Durée_exec_sec": round(elapsed, 2)
    })


# Affiche le temps dans la barre de progression
tqdm.write(f"Itération '{name}' exécutée en {elapsed:.2f} secondes")

# Afficher les résultats sous forme de tableau
results_df = pd.DataFrame(results)
# print(results_df.sort_values(by="F1-Score", ascending=False))
print(results_df.sort_values(by="F1_weighted_mean", ascending=False))

Progression:   0%|          | 0/4 [00:00<?, ?it/s]

Itération 'Naive Bayes' exécutée en 2.08 secondes
            Classifier  Accuracy_mean  Accuracy_std  F1_macro_mean  \
2                  SVM       0.898386      0.003570       0.899369   
0  Logistic Regression       0.875355      0.005216       0.874586   
1        Random Forest       0.860740      0.008259       0.860965   
3          Naive Bayes       0.831740      0.006939       0.817702   

   F1_macro_std  F1_weighted_mean  F1_weighted_std  Precision_macro_mean  \
2      0.003540          0.898826         0.003586              0.906278   
0      0.005729          0.875195         0.005379              0.882431   
1      0.008715          0.860394         0.008517              0.865907   
3      0.010107          0.827701         0.008079              0.856848   

   Recall_macro_mean  Durée_exec_sec  
2           0.897493          194.73  
0           0.873047           21.86  
1           0.861569          127.67  
3           0.812260            2.08  


**Remarques :**

En validation croisée, le score moyen nous indique la performance moyenne, mais il ne nous dit pas si cette performance est cohérente. L'écart type des scores nous donne une mesure de cette variance. Un faible écart type (< 0.05) indique que le modèle obtient des performances cohérentes sur tous les sous-ensembles de données.

**Résultats :**
Les résultats de la validation croisée confortent le résultat obtenu précédemment, à savoir SVC reste le modèle le plus performant.

